In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive
import warnings
warnings.filterwarnings('ignore')
drive.mount('/content/drive')

In [ ]:
metric_df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/PersonalFinance/final_metric_dl_new_2025.csv')
metric_df['RUN_DAY'] = pd.to_datetime(metric_df['RUN_DAY'])
final_df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/PersonalFinance/result_df_dl_new_2025.csv')
final_df['ds'] = pd.to_datetime(final_df['ds'])
final_df['RUN_DAY'] = pd.to_datetime(final_df['RUN_DAY'])

In [ ]:
metric_df.mean(numeric_only=True)

In [ ]:
# Calculate ENSMEBLE_FORECAST as the average of TFT, TFT_EXO, LSTM, and LSTM_EXO
final_df['ENSEMBLE_FORECAST'] = final_df[['TFT', 'TFT_EXO', 'LSTM', 'LSTM_EXO']].mean(axis=1)
# Calculate ENSMEBLE_BASELINE_MEDIAN as the average of TFT-median and LSTM-median
final_df['ENSEMBLE_BASELINE_MEDIAN'] = final_df[['TFT-median', 'LSTM-median']].mean(axis=1)
# Calculate ENSMEBLE_BASELINE_LO_80 as the average of TFT-lo-80 and LSTM-lo-80
final_df['ENSEMBLE_BASELINE_LO_80'] = final_df[['TFT-lo-80', 'LSTM-lo-80']].mean(axis=1)
# Calculate ENSMEBLE_BASELINE_HI_80 as the average of TFT-hi-80 and LSTM-hi-80
final_df['ENSEMBLE_BASELINE_HI_80'] = final_df[['TFT-hi-80', 'LSTM-hi-80']].mean(axis=1)
# Calculate ENSEMBLE_EXO_MEDIAN as the average of TFT-MEDIAN-EXO and LSTM-MEDIAN-EXO
final_df['ENSEMBLE_EXO_MEDIAN'] = final_df[['TFT-MEDIAN-EXO', 'LSTM-MEDIAN-EXO']].mean(axis=1)
# Calculate ENSEMBLE_EXO_LO_80 as the average of TFT-LO-80-EXO and LSTM-LO-80-EXO
final_df['ENSEMBLE_EXO_LO_80'] = final_df[['TFT-LO-80-EXO', 'LSTM-LO-80-EXO']].mean(axis=1)
# Calculate ENSEMBLE_EXO_HI_80 as the average of TFT-HI-80-EXO and LSTM-HI-80-EXO
final_df['ENSEMBLE_EXO_HI_80'] = final_df[['TFT-HI-80-EXO', 'LSTM-HI-80-EXO']].mean(axis=1)

In [ ]:
final_df

# Metric Calculation

In [ ]:
import numpy as np

In [ ]:
def smape_np(A, F):
    return 100/len(A) * np.sum(2 * np.abs(F - A) / (np.abs(A) + np.abs(F)))

In [ ]:
def wql(y_true, y_pred, quantile, weights=None):
    """
    Compute Weighted Quantile Loss (WQL) for a given quantile.

    Parameters:
        y_true (array-like): Actual values (n_samples,).
        y_pred (array-like): Predicted quantile values (n_samples,).
        quantile (float): Quantile to evaluate (e.g., 0.1 or 0.9).
        weights (array-like or None): Weights for each observation (n_samples,). If None, uniform weights are used.

    Returns:
        float: Weighted Quantile Loss.
    """
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    if weights is None:
        weights = np.ones_like(y_true)  # Uniform weights

    # Quantile Loss components
    errors = y_true - y_pred
    quantile_loss = np.maximum(quantile * errors, (quantile - 1) * errors)

    # Weighted Quantile Loss
    wql = np.sum(weights * quantile_loss) / np.sum(weights)
    return wql

In [ ]:
from scipy.stats import spearmanr

In [ ]:
final_df.columns

In [ ]:
# prompt: for each RUN_DAY, calculate the SMAPE, SPEARMANR relationship for TFT, TFT_EXO, LSTM, LSTM_EXO, TFT-median, TFT-MEDIAN-EXO, LSTM-median, LSTM-MEDIAN-EXO, ENSEMBLE_FORECAST, ENSEMBLE_BASELINE_MEDIAN, and ENSEMBLE_EXO_MEDIAN. Save the smape results to dataframe called final_metric_smape, the spearman results to final_metric_spearman

# Define the models to evaluate
models = ['TFT', 'TFT_EXO', 'LSTM', 'LSTM_EXO', 'TFT-median', 'TFT-MEDIAN-EXO', 'LSTM-median', 'LSTM-MEDIAN-EXO', 'ENSEMBLE_FORECAST', 'ENSEMBLE_BASELINE_MEDIAN', 'ENSEMBLE_EXO_MEDIAN']

# Create empty dataframes to store the results
final_metric_smape = pd.DataFrame()
final_metric_spearman = pd.DataFrame()

# Loop through each unique RUN_DAY
for day in final_df['RUN_DAY'].unique():
    temp_df = final_df[final_df['RUN_DAY'] == day]
    temp_smape = {}
    temp_spearman = {}

    # Calculate SMAPE and Spearman's rank correlation for each model
    for model in models:
        temp_smape[model] = 100 - (smape_np(temp_df['y'], temp_df[model])/2)
        temp_spearman[model], _ = spearmanr(temp_df['y'], temp_df[model])

    # Append results to the dataframes
    temp_smape_df = pd.DataFrame([temp_smape])
    temp_smape_df['RUN_DAY'] = day
    final_metric_smape = pd.concat([final_metric_smape, temp_smape_df], ignore_index=True)

    temp_spearman_df = pd.DataFrame([temp_spearman])
    temp_spearman_df['RUN_DAY'] = day
    final_metric_spearman = pd.concat([final_metric_spearman, temp_spearman_df], ignore_index=True)

In [ ]:
pd.set_option('display.max_columns', None)
final_metric_smape.mean(numeric_only=True).rename('NFA').sort_values(ascending=False)

In [ ]:
final_metric_spearman.mean(numeric_only=True)

In [ ]:
# prompt: here is an example how you get the wql score:     quantile_loss_50 = wql(Y_hat_df_temp['y'], Y_hat_df_temp['TFT-median'], 0.5)
#     quantile_loss_90 = wql(Y_hat_df_temp['y'], Y_hat_df_temp['TFT-hi-80'], 0.9)
#     quantile_loss_10 = wql(Y_hat_df_temp['y'], Y_hat_df_temp['TFT-lo-80'], 0.1)
#     avg_wql = (quantile_loss_50 + quantile_loss_90 + quantile_loss_10) / 3, help get the wql for LSTM_BASELINE_WQL, LSTM_EXO_WQL, TFT_EXO_WQL, TFT_BASELINE_WQL, ENSEMBLE_BASELINE_WQL, ENSEMBLE_EXO_WQL accordingly, save the results to final_metric_wql

final_metric_wql = pd.DataFrame()

for day in final_df['RUN_DAY'].unique():
    Y_hat_df_temp = final_df[final_df['RUN_DAY'] == day]
    temp_wql = {}

    # LSTM_BASELINE_WQL
    temp_wql['LSTM_BASELINE_WQL'] = (wql(Y_hat_df_temp['y'], Y_hat_df_temp['LSTM-median'], 0.5) +
                                    wql(Y_hat_df_temp['y'], Y_hat_df_temp['LSTM-hi-80'], 0.9) +
                                    wql(Y_hat_df_temp['y'], Y_hat_df_temp['LSTM-lo-80'], 0.1)) / 3

    # LSTM_EXO_WQL
    temp_wql['LSTM_EXO_WQL'] = (wql(Y_hat_df_temp['y'], Y_hat_df_temp['LSTM-MEDIAN-EXO'], 0.5) +
                                wql(Y_hat_df_temp['y'], Y_hat_df_temp['LSTM-HI-80-EXO'], 0.9) +
                                wql(Y_hat_df_temp['y'], Y_hat_df_temp['LSTM-LO-80-EXO'], 0.1)) / 3

    # TFT_EXO_WQL
    temp_wql['TFT_EXO_WQL'] = (wql(Y_hat_df_temp['y'], Y_hat_df_temp['TFT-MEDIAN-EXO'], 0.5) +
                               wql(Y_hat_df_temp['y'], Y_hat_df_temp['TFT-HI-80-EXO'], 0.9) +
                               wql(Y_hat_df_temp['y'], Y_hat_df_temp['TFT-LO-80-EXO'], 0.1)) / 3

    # TFT_BASELINE_WQL
    temp_wql['TFT_BASELINE_WQL'] = (wql(Y_hat_df_temp['y'], Y_hat_df_temp['TFT-median'], 0.5) +
                                   wql(Y_hat_df_temp['y'], Y_hat_df_temp['TFT-hi-80'], 0.9) +
                                   wql(Y_hat_df_temp['y'], Y_hat_df_temp['TFT-lo-80'], 0.1)) / 3

    # ENSEMBLE_BASELINE_WQL
    temp_wql['ENSEMBLE_BASELINE_WQL'] = (wql(Y_hat_df_temp['y'], Y_hat_df_temp['ENSEMBLE_BASELINE_MEDIAN'], 0.5) +
                                        wql(Y_hat_df_temp['y'], Y_hat_df_temp['ENSEMBLE_BASELINE_HI_80'], 0.9) +
                                        wql(Y_hat_df_temp['y'], Y_hat_df_temp['ENSEMBLE_BASELINE_LO_80'], 0.1)) / 3

    # ENSEMBLE_EXO_WQL
    temp_wql['ENSEMBLE_EXO_WQL'] = (wql(Y_hat_df_temp['y'], Y_hat_df_temp['ENSEMBLE_EXO_MEDIAN'], 0.5) +
                                   wql(Y_hat_df_temp['y'], Y_hat_df_temp['ENSEMBLE_EXO_HI_80'], 0.9) +
                                   wql(Y_hat_df_temp['y'], Y_hat_df_temp['ENSEMBLE_EXO_LO_80'], 0.1)) / 3

    temp_wql_df = pd.DataFrame([temp_wql])
    temp_wql_df['RUN_DAY'] = day
    final_metric_wql = pd.concat([final_metric_wql, temp_wql_df], ignore_index=True)

In [ ]:
final_metric_wql.mean(numeric_only=True).rename('WQL').sort_values(ascending=True)

In [ ]:
final_df

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd

# Prepare and aggregate
plot_df = final_df.copy()
plot_df['RUN_DAY'] = pd.to_datetime(plot_df['RUN_DAY'])
plot_df.sort_values(['RUN_DAY', 'ds', 'unique_id'], inplace=True)

agg_df = plot_df.groupby('RUN_DAY').agg({
    'y': 'sum',
    'LSTM-MEDIAN-EXO': 'sum',
    'LSTM-LO-80-EXO': 'sum',
    'LSTM-HI-80-EXO': 'sum'
}).reset_index()

# Set Seaborn theme
sns.set_theme(style="whitegrid", font_scale=1.2)

# Plot
plt.figure(figsize=(18, 6), dpi=300)
sns.lineplot(data=agg_df, x='RUN_DAY', y='y', label='Observed', color='black')
sns.lineplot(data=agg_df, x='RUN_DAY', y='LSTM-MEDIAN-EXO', label='LSTM Median Forecast', color='royalblue', linestyle='--')

plt.fill_between(agg_df['RUN_DAY'],
                 agg_df['LSTM-LO-80-EXO'],
                 agg_df['LSTM-HI-80-EXO'],
                 alpha=0.25, color='skyblue', label='80% Prediction Interval')

# Formatting
plt.title('Total Forecasted EUI Over 30-Day Horizon', fontsize=18, weight='bold')
plt.xlabel('Run Day')
plt.ylabel('Total EUI')
plt.xticks(rotation=45)
plt.gca().xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))

plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd

plot_df = final_df.copy()
plot_df['RUN_DAY'] = pd.to_datetime(plot_df['RUN_DAY'])
plot_df.sort_values(['RUN_DAY', 'ds', 'unique_id'], inplace=True)

agg_df = plot_df.groupby('RUN_DAY').agg({
    'y': 'sum',
    'LSTM-MEDIAN-EXO': 'sum',
    'LSTM-LO-80-EXO': 'sum',
    'LSTM-HI-80-EXO': 'sum'
}).reset_index()

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=agg_df['RUN_DAY'], y=agg_df['y'],
    mode='lines', name='Observed',
    line=dict(color='black', width=2)
))

fig.add_trace(go.Scatter(
    x=agg_df['RUN_DAY'], y=agg_df['LSTM-MEDIAN-EXO'],
    mode='lines', name='LSTM Median Forecast',
    line=dict(color='royalblue', width=2, dash='dash')
))

fig.add_trace(go.Scatter(
    x=agg_df['RUN_DAY'], y=agg_df['LSTM-LO-80-EXO'],
    mode='lines', name='Lower Bound (80%)',
    line=dict(color='skyblue'), showlegend=False
))

fig.add_trace(go.Scatter(
    x=agg_df['RUN_DAY'], y=agg_df['LSTM-HI-80-EXO'],
    fill='tonexty', name='80% Prediction Interval',
    line=dict(color='skyblue'), mode='lines',
    fillcolor='rgba(135,206,250,0.2)'
))

fig.update_layout(
    title='Total Forecasted EUI Over 30-Day Horizon',
    xaxis_title='Run Day',
    yaxis_title='Total EUI',
    template='plotly_white',
    font=dict(size=14),
    legend=dict(x=0.01, y=0.99),
    margin=dict(t=50, l=50, r=50, b=50)
)

fig.show()


In [ ]:
import matplotlib.pyplot as plt

# Feature importance data
features = {
    'Retirement Investment Options': 0.12049467187178763,
    'Buying and Selling of Financial Products': 0.11386940604762028,
    'Medical Billing and Collections': 0.14636619969418174,
    'Social Security Benefits': 0.11888130589535362,
    'Stock Market': 0.115982396979081,
    'Observed Target': 0.18349468833521793,
    'Repeated Target': 0.2009113311767578
}

# Sort features by importance, highest first
features = dict(sorted(features.items(), key=lambda item: item[1], reverse=True))

feature_names = list(features.keys())
importance_values = list(features.values())

plt.figure(figsize=(21, 7), dpi=300)
bars = plt.barh(feature_names, importance_values, color='skyblue', edgecolor='black')

for i, bar in enumerate(bars):
    name = feature_names[i]
    if name == 'Repeated Target' or name == 'Observed Target':
        bar.set_color('lightgray')
        bar.set_edgecolor('dimgray')
    elif name == 'Medical Billing and Collections':
        bar.set_color('steelblue')
        bar.set_edgecolor('navy')
    else:
        bar.set_color('skyblue')
        bar.set_edgecolor('black')

plt.xlabel('Importance Score', fontsize=14, fontweight='bold')
plt.title('Feature Importances Based on Average Attention Weights From TFT',
          fontsize=16, fontweight="bold", pad=20)
plt.gca().invert_yaxis()  # Most important feature at the top
plt.grid(axis='x', linestyle='--', alpha=0.6)

# Set bold y-axis labels (feature names)
plt.yticks(fontsize=13, fontweight='bold')

# Value labels on each bar
for bar in bars:
    width = bar.get_width()
    plt.text(width + 0.003, bar.get_y() + bar.get_height() / 2,
             f'{width:.3f}', va='center', ha='left', fontsize=12)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Permutation feature importance data (LSTM)
features = {
    'Retirement Investment Options': 0.093,
    'Buying and Selling of Financial Products': 0.085,
    'Medical Billing and Collections': 0.142,
    'Stock Market': 0.110,
    'Alcoholic Beverage Consumption': 0.075
}

# Sort features by importance, highest first
features = dict(sorted(features.items(), key=lambda item: item[1], reverse=True))

feature_names = list(features.keys())
importance_values = list(features.values())

plt.figure(figsize=(21, 7), dpi=300)
bars = plt.barh(feature_names, importance_values, color='salmon', edgecolor='black')

# Optional: highlight the most important feature
bars[0].set_color('firebrick')
bars[0].set_edgecolor('maroon')

plt.xlabel('Permutation Importance Score', fontsize=14, fontweight='bold')
plt.title('Permutation Feature Importances of LSTM',
          fontsize=16, fontweight="bold", pad=20)
plt.gca().invert_yaxis()  # Most important feature at the top
plt.grid(axis='x', linestyle='--', alpha=0.6)

# Set bold y-axis labels (feature names)
plt.yticks(fontsize=13, fontweight='bold')

# Value labels on each bar
for bar in bars:
    width = bar.get_width()
    plt.text(width + 0.0015, bar.get_y() + bar.get_height() / 2,
             f'{width:.3f}', va='center', ha='left', fontsize=12)

plt.tight_layout()
plt.show()
